# Keras EfficientNetB3 ordinal semi-supervised

Notebook này dùng trực tiếp model thầy cung cấp tại `MyDrive/best_EfficientNetB3_rgb_crop_v1.keras`. Luồng chạy: kiểm tra model → tải labeled/unlabeled từ Kaggle → đánh giá baseline → tạo pseudo-label và fine-tune → test checkpoint tốt nhất.

Training lưu `BackupAndRestore`, manifests và checkpoints trên Drive. Nếu Colab ngắt kết nối, chạy lại notebook với `RESUME = True` và giữ nguyên `RUN_NAME`. Test split không tham gia training hoặc chọn checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive')
MODEL_PATH = DRIVE_ROOT / 'best_EfficientNetB3_rgb_crop_v1.keras'
RUN_NAME = 'keras_ordinal_semi_v1'  # Giữ nguyên tên này để resume
OUTPUT_DIR = DRIVE_ROOT / 'keras_semi_runs' / RUN_NAME

GITHUB_USERNAME = 'Bang334'
GITHUB_REPO = 'dr-diagnostic-system'
GITHUB_BRANCH = 'feat/keras-grade-semi-supervised'

LABELED_KAGGLE = 'sehastrajits/fundus-aptosddridirdeyepacsmessidor'
UNLABELED_KAGGLE = 'griffchristenson/unlabeled-retinal-image-dataset'

RUN_BASELINE = True
RUN_TRAINING = True
RESUME = True  # An toàn cả lần đầu; lần sau phục hồi training-backup
RUN_FINAL_TEST = False  # Chỉ bật sau khi đã chọn xong checkpoint

EPOCHS = 8
BATCH_SIZE = 8
PSEUDO_CONFIDENCE = 0.50
PSEUDO_WEIGHT = 0.25
MAX_PSEUDO_PER_CLASS = 0
MAX_UNLABELED_IMAGES = 0
LOG_EVERY_BATCHES = 25
SHOW_TENSORBOARD = False

assert MODEL_PATH.is_file(), f'Không tìm thấy model: {MODEL_PATH}'
print('Model :', MODEL_PATH)
print('Output:', OUTPUT_DIR)

## 1. Lấy đúng branch và cài dependencies
Branch phải được push lên GitHub trước khi chạy notebook. Cell có thể chạy lại sau khi Colab reconnect.

In [ ]:
import os, shutil, subprocess, sys
REPO_DIR = Path('/content') / GITHUB_REPO
REPO_URL = f'https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'
if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'switch', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ai/semi_supervised/requirements-keras.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kaggle>=2.2.2'], check=True)
print('Repository:', REPO_DIR)

## 2. Tải đúng dữ liệu từ Kaggle

Thêm `KAGGLE_API_TOKEN` trong Colab Secrets. Labeled replay dùng bộ fundus gộp APTOS/DDR/IDRiD/EyePACS/Messidor; unlabeled pool dùng bộ ảnh võng mạc chưa nhãn. Hai nguồn phải khác nhau.

In [ ]:
import getpass, zipfile
from google.colab import userdata

def kaggle_token():
    try:
        token = userdata.get('KAGGLE_API_TOKEN')
    except Exception:
        token = getpass.getpass('Kaggle API token: ').strip()
    if not token:
        raise RuntimeError('Chưa cung cấp KAGGLE_API_TOKEN')
    os.environ['KAGGLE_API_TOKEN'] = token

def download_kaggle(dataset_ref, name):
    target = Path('/content/keras_semi_data') / name
    marker = target / '.kaggle_source'
    if marker.is_file() and marker.read_text().strip() == dataset_ref:
        print('Dùng lại:', target)
        return target
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True)
    kaggle_token()
    subprocess.run([sys.executable, '-m', 'kaggle', 'datasets', 'download', '-d', dataset_ref, '-p', str(target)], check=True)
    archives = list(target.glob('*.zip'))
    if not archives:
        raise FileNotFoundError(f'Kaggle không tạo ZIP cho {dataset_ref}')
    for archive_path in archives:
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(target)
        archive_path.unlink()
    marker.write_text(dataset_ref)
    return target

LABELED_ROOT = download_kaggle(LABELED_KAGGLE, 'labeled_merged')
UNLABELED_ROOT = download_kaggle(UNLABELED_KAGGLE, 'unlabeled_retina')
from ai.semi_supervised.keras_semi_supervised import find_splits, discover_images
SPLITS = find_splits(LABELED_ROOT)
DATASET_DIR = next(iter(SPLITS.values())).parent
print('Splits:', SPLITS)
print('Unlabeled images:', len(discover_images(UNLABELED_ROOT)))

## 3. Test trực tiếp model trên Drive
Cell này load file `.keras`, kiểm tra input/output và dự đoán một ảnh validation bằng TTA.

In [ ]:
from ai.keras_grading.grader import KerasOrdinalGrader
from ai.keras_grading.evaluate import scan_class_folder
grader = KerasOrdinalGrader(MODEL_PATH, use_tta=True)
print('Load mode   :', grader.load_mode)
print('Input shape :', grader.model.input_shape)
print('Output shape:', grader.model.output_shape)
print('Thresholds  :', grader.thresholds.tolist())
sample_path, sample_truth = scan_class_folder(SPLITS['val'])[0]
sample_prediction = grader.predict_path(sample_path)
print('Sample:', sample_path)
print('Truth :', sample_truth)
print('Predict:', sample_prediction.as_dict())

## 4. Baseline trên validation
Baseline được lưu riêng trên Drive. Nếu đã có kết quả thì cell chỉ đọc lại, không chạy lại toàn bộ.

In [ ]:
import json
BASELINE_DIR = OUTPUT_DIR / 'teacher-validation'
metrics_path = BASELINE_DIR / 'metrics.json'
if RUN_BASELINE and not metrics_path.is_file():
    subprocess.run([sys.executable, '-u', '-m', 'ai.keras_grading.evaluate', '--model', str(MODEL_PATH), '--split-dir', str(SPLITS['val']), '--output-dir', str(BASELINE_DIR)], check=True)
if metrics_path.is_file():
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2, ensure_ascii=False))

## 5. Semi-supervised train hoặc resume

- `training-backup/`: trạng thái epoch, model và optimizer để phục hồi sau ngắt kết nối.
- `pseudo_labels.csv`: pseudo-label cố định, không sinh lại khi cache còn hợp lệ.
- `pseudo-cache.json`: lưu model, inventory ảnh, TTA và các ngưỡng đã dùng. Cùng cấu hình sẽ đọc lại từ Drive; đổi ngưỡng sẽ tự tạo lại.
- `checkpoint-best.keras`: checkpoint tốt nhất theo validation QWK.
- Chạy lại từ đầu notebook với cùng `RUN_NAME` và `RESUME=True` để tiếp tục.

In [ ]:
TRAIN_DIR = OUTPUT_DIR / 'training'
command = [
    sys.executable, '-u', '-m', 'ai.semi_supervised.keras_semi_supervised',
    '--model', str(MODEL_PATH),
    '--dataset-dir', str(DATASET_DIR),
    '--unlabeled-dir', str(UNLABELED_ROOT),
    '--output-dir', str(TRAIN_DIR),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--pseudo-confidence', str(PSEUDO_CONFIDENCE),
    '--pseudo-weight', str(PSEUDO_WEIGHT),
    '--max-pseudo-per-class', str(MAX_PSEUDO_PER_CLASS),
    '--max-unlabeled-images', str(MAX_UNLABELED_IMAGES),
    '--log-every-batches', str(LOG_EVERY_BATCHES),
]
if RESUME:
    command.append('--resume')
print(' '.join(command))
if RUN_TRAINING:
    subprocess.run(command, check=True)

## 6. Xem artifacts và phân bố pseudo-label

In [ ]:
import pandas as pd
pseudo_path = TRAIN_DIR / 'pseudo_labels.csv'
if pseudo_path.is_file():
    pseudo = pd.read_csv(pseudo_path)
    display(pseudo.head())
    display(pseudo.groupby('diagnosis').agg(images=('image_path', 'size'), mean_confidence=('confidence', 'mean')))
history_path = TRAIN_DIR / 'history.csv'
if history_path.is_file():
    display(pd.read_csv(history_path).tail(10))
for name in ('pseudo_labels.csv', 'pseudo-cache.json', 'history.csv', 'epoch-log.json', 'run.json', 'checkpoint-best.keras', 'checkpoint-last.keras', 'tensorboard'):
    path = TRAIN_DIR / name
    print(name, 'OK' if path.exists() else 'chưa có')
if SHOW_TENSORBOARD and (TRAIN_DIR / 'tensorboard').is_dir():
    from tensorboard import notebook as tb_notebook
    tb_notebook.start(f'--logdir {TRAIN_DIR / "tensorboard"}')

## 7. Final test — chỉ chạy sau khi kết thúc lựa chọn model
Bật `RUN_FINAL_TEST=True` ở cell cấu hình. Mỗi checkpoint chỉ được test một lần; output tồn tại sẽ không bị ghi đè.

In [ ]:
BEST_MODEL = TRAIN_DIR / 'checkpoint-best.keras'
FINAL_TEST_DIR = OUTPUT_DIR / 'final-test-best'
if RUN_FINAL_TEST:
    if not BEST_MODEL.is_file():
        raise FileNotFoundError(BEST_MODEL)
    if (FINAL_TEST_DIR / 'metrics.json').is_file():
        print('Final test đã tồn tại; không chạy lại:', FINAL_TEST_DIR)
    else:
        subprocess.run([sys.executable, '-u', '-m', 'ai.keras_grading.evaluate', '--model', str(BEST_MODEL), '--split-dir', str(SPLITS['test']), '--output-dir', str(FINAL_TEST_DIR)], check=True)
        print((FINAL_TEST_DIR / 'metrics.json').read_text())
else:
    print('Final test đang khóa. Đặt RUN_FINAL_TEST=True khi training và model selection đã hoàn tất.')

## 8. Dự đoán một ảnh tùy chọn từ Drive

In [ ]:
CUSTOM_IMAGE = ''  # Ví dụ: '/content/drive/MyDrive/test_fundus.jpg'
selected_model = BEST_MODEL if BEST_MODEL.is_file() else MODEL_PATH
if CUSTOM_IMAGE:
    custom_grader = KerasOrdinalGrader(selected_model, use_tta=True)
    print(custom_grader.predict_path(CUSTOM_IMAGE).as_dict())
else:
    print('Đặt CUSTOM_IMAGE để dự đoán một ảnh riêng.')